In [1]:
!pip install numpy nltk scikit-learn transformers matplotlib seaborn pandas sentence-transformers datasets

In [ ]:

import os
import random
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForCausalLM
import torch
from sentence_transformers import SentenceTransformer

# Ensure NLTK packages are downloaded
nltk.download('punkt')


set_seed(42)
random.seed(42)
np.random.seed(42)

# Define the prompts
prompts = [
    # 1.Philosophical Question:
    "What is the meaning of true happiness in life?",

    # 2.Hypothetical Scenario:
    "If humans could live on Mars, what challenges would they face and how could they overcome them?",

    # 3.Creative Thinking Prompt:
    "Can you describe an imaginary city where technology and nature exist in perfect harmony?",

    # 4.Practical Advice Question:
    "What are the most effective ways to learn a new language quickly?",

    # 5.Exploration of Abstract Concepts:
    "How would you explain the concept of time to someone who has never experienced it?",
    
    # 6.Scientific Exploration:
    "What are the possible effects of artificial intelligence on scientific research in the next decade?",

    # 7.Ethical Dilemma:
    "Is it ever justifiable to prioritize technological advancement over environmental protection?",

    # 8.Problem-Solving Question:
    "How can cities effectively reduce traffic congestion without compromising accessibility?",

    # 9.Imaginative Scenario:
    "If animals could communicate with humans, how would that change our world?",

    # 10.Personal Reflection Prompt:
    "What qualities make someone a great leader, and how can those qualities be developed?"
    ]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

num_samples = 100 # 100, 250, 500, 1000, 1500, 2000
max_length = 125

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model_name = "gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.to(device)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1  
)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
print(next(model.parameters()).device)  

cuda:0


In [ ]:
def generate_outputs(prompt, num_samples=num_samples, temperature=0.9):
    outputs = []
    for _ in range(num_samples):
        response = generator(
            prompt,
            max_length=max_length,
            num_return_sequences=1,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
            pad_token_id=generator.tokenizer.eos_token_id,
            truncation=True  
        )
        outputs.append(response[0]['generated_text'])
    return outputs

def remove_prompt_from_output(prompt, output):
    if output.startswith(prompt):
        return output[len(prompt):].strip()
    else:
        return output.strip()


all_outputs = {}
for prompt in prompts:
    outputs = generate_outputs(prompt)
    outputs = [remove_prompt_from_output(prompt, output) for output in outputs]
    all_outputs[prompt] = outputs

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
import json

output_file = "gpt2_100125.json"
with open(output_file, "w") as f:
    json.dump(all_outputs, f)

print(f"All outputs saved to {output_file}")

In [ ]:
import random  

for prompt, outputs in all_outputs.items():
    print(f"Prompt: {prompt}")
    random_outputs = random.sample(outputs, 3)
    for idx, output in enumerate(random_outputs):
        print(f"Output {idx+1}: {output}")
    print("-" * 80)